# Cellpose Batch Analysis — Two-Channel + Condition Comparison

Segments cells on the **GFP (CH2)** channel, measures both CH2 and far-red (CH4) intensity
per cell, and compares the CH4/GFP ratio across experimental conditions.

---

### Quick-start
1. **Runtime → Change runtime type → T4 GPU** (recommended)
2. Run **Cell 1** to install packages
3. Run **Cell 2** to mount Google Drive
4. **Edit Cell 3 only** — set your folder paths and conditions
5. Run all remaining cells in order

---

### Expected Google Drive folder structure
```
MyDrive/
└── YourExperiment/
    ├── Splus_Bplus/          ← one subfolder per condition
    │   ├── image1_CH2.tif
    │   ├── image1_CH4.tif
    │   └── image2_CH2.tif
    │   └── image2_CH4.tif
    └── Sminus_Bplus/
        ├── image1_CH2.tif
        └── image1_CH4.tif
```
CH2 and CH4 files are paired by matching filename — every `_CH2.tif` must have a
corresponding `_CH4.tif` with the same prefix.

## Cell 1 — Install dependencies
Run once per Colab session.

In [1]:
%pip install cellpose scikit-image matplotlib seaborn pandas natsort tqdm --quiet
print("Packages ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 59.9 MB/s eta 0:00:00:00:0100:01
Packages ready


## Cell 2 — Mount Google Drive

In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive")
else:
    print("Running locally — Drive mount skipped")

Mounted at /content/drive
Google Drive mounted at /content/drive


## Cell 3 — Configuration
**This is the only cell you need to edit.**

In [ ]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
# Root folder that contains one subfolder per condition
BASE_DIR = Path("/content/drive/MyDrive/DATA/Chris/01122026_psychtag34")

# Names of the condition subfolders inside BASE_DIR
CONDITIONS = ["Control", "Drug"]

# Folder where results CSV and mask TIFFs will be saved
SAVE_DIR = BASE_DIR / "results"

# ── File mode ─────────────────────────────────────────────────────────────────
# "separate_tif"  — two files per image: one *_CH2.tif and one *_CH4.tif
# "ome_tiff"      — single multi-channel .ome.tiff file per image
FILE_MODE = "separate_tif"

# ── OME-TIFF channel indices (only used when FILE_MODE = "ome_tiff") ──────────
# Channels are 0-indexed:  ch1 = 0,  ch2 = 1,  ch3 = 2, etc.
REF_CHANNEL_IDX = 1   # ch2 — baseline / segmentation channel (Cellpose runs on this)
SIG_CHANNEL_IDX = 0   # ch1 — expression / signal channel

# ── Separate-TIF file naming (only used when FILE_MODE = "separate_tif") ──────
CH2_TAG  = "CH2"   # substring in the baseline/GFP filename
CH4_TAG  = "CH4"   # substring in the signal/far-red filename
FILE_EXT = ".tif"

# ── Cellpose parameters ───────────────────────────────────────────────────────
# Model options: "cyto3" (default), "cyto2", "nuclei"
CELLPOSE_MODEL     = "cyto3"
CELL_DIAMETER      = 0      # approximate cell diameter in pixels (0 = auto-estimate)
FLOW_THRESHOLD     = 0.4
CELLPROB_THRESHOLD = -2.0   # lower = more permissive (try -4.0 if still missing cells)
USE_GPU            = True   # set False if no GPU

# ── Cell size filtering ───────────────────────────────────────────────────────
MIN_CELL_AREA_PX = 100
MAX_CELL_AREA_PX = 10000

# ── Sanity check ─────────────────────────────────────────────────────────────
for cond in CONDITIONS:
    cond_dir = BASE_DIR / cond
    status = "OK" if cond_dir.exists() else "NOT FOUND"
    if FILE_MODE == "ome_tiff":
        files = sorted(cond_dir.rglob("*.ome.tiff")) if cond_dir.exists() else []
        print(f"  [{status}]  {cond_dir}  ({len(files)} .ome.tiff files)")
    else:
        ch2_files = sorted(cond_dir.rglob(f"*{CH2_TAG}*{FILE_EXT}")) if cond_dir.exists() else []
        print(f"  [{status}]  {cond_dir}  ({len(ch2_files)} CH2 files)")

## Cell 3b — Image diagnostic (optional)
Run this to verify your images are loading correctly before the full batch. Check that **Max** is not 0 and the image looks as expected.

In [ ]:
# ── Diagnostic: inspect one image before running the full batch ───────────────
import tifffile as _tifffile
import matplotlib.pyplot as _plt
import numpy as _np
from skimage import io as _io

_cond_dir = BASE_DIR / CONDITIONS[0]

if FILE_MODE == "ome_tiff":
    _files = sorted(_cond_dir.rglob("*.ome.tiff"))
    if not _files:
        print("No .ome.tiff files found — check BASE_DIR and FILE_MODE")
    else:
        _img = _tifffile.imread(str(_files[0])).astype(_np.float32).squeeze()
        print(f"File  : {_files[0].name}")
        print(f"Shape : {_img.shape}   ← expected (C, H, W) or (H, W, C)")
        print(f"Dtype : {_img.dtype}")
        n_ch = _img.shape[0] if (_img.ndim == 3 and _img.shape[0] <= 8) else _img.shape[2]
        print(f"Channels available: {n_ch}  (indices 0 … {n_ch-1})")
        print(f"REF_CHANNEL_IDX={REF_CHANNEL_IDX} (ch{REF_CHANNEL_IDX+1}, baseline/segmentation)")
        print(f"SIG_CHANNEL_IDX={SIG_CHANNEL_IDX} (ch{SIG_CHANNEL_IDX+1}, expression/signal)")

        _fig, _axes = _plt.subplots(1, n_ch, figsize=(5 * n_ch, 5))
        if n_ch == 1:
            _axes = [_axes]
        for _i, _ax in enumerate(_axes):
            _ch = _img[_i] if _img.shape[0] <= 8 else _img[..., _i]
            _p2, _p98 = _np.percentile(_ch, (2, 98))
            _ax.imshow(_ch, cmap="gray", vmin=_p2, vmax=_p98)
            _label = ""
            if _i == REF_CHANNEL_IDX: _label = " ← REF (baseline)"
            if _i == SIG_CHANNEL_IDX: _label = " ← SIG (expression)"
            _ax.set_title(f"ch{_i+1} (idx {_i}){_label}\nmax={_ch.max():.0f}", fontsize=9)
            _ax.axis("off")
        _plt.suptitle(f"Diagnostic: {_files[0].name}", fontsize=10)
        _plt.tight_layout()
        _plt.show()
else:
    _pairs = sorted(_cond_dir.rglob(f"*{CH2_TAG}*{FILE_EXT}"))
    _pairs = [f for f in _pairs if not any(t in f.name for t in ("_masks", "_overlay", "_seg"))]
    if not _pairs:
        print("No CH2 images found — check BASE_DIR and CH2_TAG")
    else:
        _img = _io.imread(str(_pairs[0]))
        print(f"File  : {_pairs[0].name}")
        print(f"Shape : {_img.shape}")
        print(f"Dtype : {_img.dtype}")
        print(f"Min   : {_img.min()},  Max: {_img.max()},  Mean: {_img.mean():.1f}")
        _show = _img[..., 0] if _img.ndim == 3 else _img
        _p2, _p98 = _np.percentile(_show, (2, 98))
        print(f"Display range: {_p2:.0f} – {_p98:.0f}  (2nd–98th percentile)")
        _plt.figure(figsize=(6, 6))
        _plt.imshow(_show, cmap="gray", vmin=_p2, vmax=_p98)
        _plt.colorbar(label="intensity")
        _plt.title(f"Raw CH2 — {_pairs[0].name}")
        _plt.axis("off")
        _plt.tight_layout()
        _plt.show()

## Cell 4 — Imports and Cellpose model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tifffile
from skimage import io
from skimage.measure import regionprops_table
from cellpose import models, plot
from tqdm.notebook import tqdm

import torch
gpu_ok = USE_GPU and torch.cuda.is_available()
print(f"GPU available: {torch.cuda.is_available()} | Using GPU: {gpu_ok}")

model = models.CellposeModel(gpu=gpu_ok, model_type=CELLPOSE_MODEL)
print(f"Cellpose model loaded  (model: {CELLPOSE_MODEL})")
print(f"File mode: {FILE_MODE}")

## Cell 5 — Helper functions

In [ ]:
def _load_ome_channel(path: Path, channel_idx: int) -> np.ndarray:
    """
    Extract one channel from a multi-channel .ome.tiff as a 2-D float array.
    Supports (C, H, W) and (H, W, C) layouts.
    """
    img = tifffile.imread(str(path)).astype(float)
    img = img.squeeze()
    if img.ndim == 3:
        if img.shape[0] <= 8:       # (C, H, W)
            return img[channel_idx]
        elif img.shape[2] <= 8:     # (H, W, C)
            return img[..., channel_idx]
        else:
            raise ValueError(f"Cannot determine channel axis for shape {img.shape} in {path.name}")
    return img   # already 2-D


def find_image_pairs(condition_dir: Path):
    """
    Returns a list of (ref_path, sig_path) tuples for all images in condition_dir.

    FILE_MODE = "ome_tiff"    — each .ome.tiff contains all channels; ref and sig
                                 paths are the same file (channels extracted by index).
    FILE_MODE = "separate_tif" — paired *_CH2.tif / *_CH4.tif files.
    """
    if FILE_MODE == "ome_tiff":
        files = sorted(condition_dir.rglob("*.ome.tiff"))
        files = [f for f in files if not any(t in f.name for t in ("_masks", "_overlay", "_seg"))]
        if not files:
            print(f"  WARNING: no .ome.tiff files found in {condition_dir}")
        return [(f, f) for f in files]
    else:
        pairs = []
        ch2_files = sorted(condition_dir.rglob(f"*{CH2_TAG}*{FILE_EXT}"))
        ch2_files = [f for f in ch2_files if not any(tag in f.name for tag in ("_masks", "_overlay", "_seg"))]
        for ch2 in ch2_files:
            ch4_name = ch2.name.replace(CH2_TAG, CH4_TAG)
            ch4 = ch2.parent / ch4_name
            if ch4.exists():
                pairs.append((ch2, ch4))
            else:
                print(f"  WARNING: no CH4 match found for {ch2.name} — skipping")
        return pairs


def analyze_image_pair(ref_path: Path, sig_path: Path, condition: str, replicate: str):
    """
    Segments cells on the reference channel, measures per-cell mean intensity
    in both reference and signal channels. Filters by MIN/MAX_CELL_AREA_PX.

    Returns a DataFrame with columns:
      condition, replicate, cell_id, area_px, mean_ref, mean_sig
    """
    try:
        if FILE_MODE == "ome_tiff":
            ref_img = _load_ome_channel(ref_path, REF_CHANNEL_IDX)
            sig_img = _load_ome_channel(sig_path, SIG_CHANNEL_IDX)
        else:
            ref_raw = io.imread(str(ref_path)).astype(float)
            sig_raw = io.imread(str(sig_path)).astype(float)
            ref_img = ref_raw[..., 0] if ref_raw.ndim == 3 else ref_raw
            sig_img = sig_raw[..., 0] if sig_raw.ndim == 3 else sig_raw
    except Exception as e:
        print(f"  ERROR loading {ref_path.name}: {e} — skipping")
        return pd.DataFrame()

    try:
        masks, _, _ = model.eval(
            [ref_img],
            diameter=CELL_DIAMETER,
            flow_threshold=FLOW_THRESHOLD,
            cellprob_threshold=CELLPROB_THRESHOLD,
        )
    except Exception as e:
        print(f"  ERROR segmenting {ref_path.name}: {e} — skipping")
        return pd.DataFrame()

    mask = masks[0]
    n_cells = mask.max()
    print(f"    {ref_path.name}: {n_cells} cell(s) detected")

    if n_cells == 0:
        return pd.DataFrame()

    props_ref = regionprops_table(mask, intensity_image=ref_img,
                                   properties=["label", "area", "mean_intensity"])
    props_sig = regionprops_table(mask, intensity_image=sig_img,
                                   properties=["label", "mean_intensity"])

    df = pd.DataFrame({
        "condition": condition,
        "replicate": replicate,
        "cell_id":   props_ref["label"],
        "area_px":   props_ref["area"],
        "mean_ref":  props_ref["mean_intensity"],
        "mean_sig":  props_sig["mean_intensity"],
    })

    before = len(df)
    df = df[(df["area_px"] >= MIN_CELL_AREA_PX) & (df["area_px"] <= MAX_CELL_AREA_PX)]
    excluded = before - len(df)
    if excluded:
        print(f"      → {excluded} cell(s) excluded by area filter ({MIN_CELL_AREA_PX}–{MAX_CELL_AREA_PX} px²)")

    return df


print("Helper functions defined")

## Cell 6 — Run batch analysis
Processes all image pairs across every condition folder.

In [ ]:
import datetime

SAVE_DIR.mkdir(parents=True, exist_ok=True)
overlay_dir = SAVE_DIR / "overlays"
overlay_dir.mkdir(parents=True, exist_ok=True)

all_rows = []
run_start = datetime.datetime.now()

for condition in CONDITIONS:
    cond_dir = BASE_DIR / condition
    if not cond_dir.exists():
        print(f"SKIP: {cond_dir} does not exist")
        continue

    pairs = find_image_pairs(cond_dir)
    print(f"\nCondition: {condition} — {len(pairs)} image(s)")

    for ref_path, sig_path in tqdm(pairs, desc=condition, unit="image"):
        if FILE_MODE == "ome_tiff":
            replicate = ref_path.stem
        else:
            replicate = ref_path.stem.replace(CH2_TAG, "").strip("_- ")

        df_pair = analyze_image_pair(ref_path, sig_path, condition, replicate)
        if not df_pair.empty:
            all_rows.append(df_pair)

        # Save segmentation overlay PNG for QC
        try:
            if FILE_MODE == "ome_tiff":
                _seg = _load_ome_channel(ref_path, REF_CHANNEL_IDX)
            else:
                _raw = io.imread(str(ref_path)).astype(float)
                _seg = _raw[..., 0] if _raw.ndim == 3 else _raw

            _masks_ov, _flows_ov, _ = model.eval(
                [_seg],
                diameter=CELL_DIAMETER,
                flow_threshold=FLOW_THRESHOLD,
                cellprob_threshold=CELLPROB_THRESHOLD,
            )
            _fig = plt.figure(figsize=(14, 4))
            plot.show_segmentation(_fig, _seg, _masks_ov[0], _flows_ov)
            _out = overlay_dir / f"{ref_path.stem}_overlay.png"
            _fig.savefig(str(_out), dpi=100, bbox_inches="tight")
            plt.close(_fig)
        except Exception as _e:
            print(f"  WARNING: could not save overlay for {ref_path.name}: {_e}")

if all_rows:
    df_all = pd.concat(all_rows, ignore_index=True)
    print(f"\nTotal cells measured: {len(df_all)}")
    display(df_all.head())
else:
    df_all = pd.DataFrame(columns=["condition", "replicate", "cell_id",
                                    "area_px", "mean_ref", "mean_sig"])
    print("No cells detected — check BASE_DIR and file naming.")

## Cell 7 — Compute CH4/GFP ratio

In [ ]:
if df_all.empty:
    print("No data — run Cell 6 first.")
else:
    df_all["ratio_sig_ref"] = df_all["mean_sig"] / df_all["mean_ref"].replace(0, float("nan"))

    stats = df_all.groupby("condition")[["mean_ref", "mean_sig", "ratio_sig_ref"]].describe().round(2)
    print(stats)

    # Per-condition summary CSV
    summary = df_all.groupby("condition").agg(
        n_cells        = ("cell_id", "count"),
        n_images       = ("replicate", "nunique"),
        mean_ratio     = ("ratio_sig_ref", "mean"),
        std_ratio      = ("ratio_sig_ref", "std"),
        median_ratio   = ("ratio_sig_ref", "median"),
        mean_ref       = ("mean_ref", "mean"),
        mean_sig       = ("mean_sig", "mean"),
    ).round(3).reset_index()

    summary_path = SAVE_DIR / "summary.csv"
    summary.to_csv(summary_path, index=False)
    print(f"\nPer-condition summary saved to {summary_path}")
    display(summary)

## Cell 8 — Violin plot: ratio by condition

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.violinplot(
    data=df_all,
    x="condition",
    y="ratio_sig_ref",
    inner="box",
    palette="Set2",
    ax=ax,
)
ax.set_title("Signal / Baseline intensity ratio per condition")
ax.set_xlabel("Condition")
ax.set_ylabel("Signal / Baseline ratio")
plt.tight_layout()

violin_path = SAVE_DIR / "violin_ratio.png"
fig.savefig(str(violin_path), dpi=150, bbox_inches="tight")
print(f"Violin plot saved to {violin_path}")
plt.show()

## Cell 9 — Scatter plot: GFP vs far-red intensity

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=df_all,
    x="mean_ref",
    y="mean_sig",
    hue="condition",
    alpha=0.6,
    palette="Set2",
    ax=ax,
)
ax.set_title("Baseline vs signal intensity per cell")
ax.set_xlabel("Mean baseline intensity (ref channel)")
ax.set_ylabel("Mean signal intensity (sig channel)")
plt.tight_layout()

scatter_path = SAVE_DIR / "scatter_ref_vs_sig.png"
fig.savefig(str(scatter_path), dpi=150, bbox_inches="tight")
print(f"Scatter plot saved to {scatter_path}")
plt.show()

## Cell 10 — Segmentation overlay (first image)

In [ ]:
# Re-segment the first image of the first condition for display
first_cond = CONDITIONS[0]
pairs_preview = find_image_pairs(BASE_DIR / first_cond)

if pairs_preview:
    ref_path, _ = pairs_preview[0]

    if FILE_MODE == "ome_tiff":
        seg_preview = _load_ome_channel(ref_path, REF_CHANNEL_IDX)
    else:
        img_preview = io.imread(str(ref_path)).astype(float)
        seg_preview = img_preview[..., 0] if img_preview.ndim == 3 else img_preview

    masks_p, flows_p, _ = model.eval(
        [seg_preview],
        diameter=CELL_DIAMETER,
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
    )

    fig = plt.figure(figsize=(14, 4))
    plot.show_segmentation(fig, seg_preview, masks_p[0], flows_p)
    plt.suptitle(f"Segmentation overlay — {ref_path.name}", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print(f"No images found in {first_cond} for preview.")

## Cell 11 — Save results to CSV

In [ ]:
import datetime

SAVE_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
csv_path = SAVE_DIR / f"results_{timestamp}.csv"
df_all.to_csv(csv_path, index=False)
print(f"Saved {len(df_all)} rows to {csv_path}")

log_path = SAVE_DIR / "run_log.txt"
run_end = datetime.datetime.now()
with open(log_path, "a") as _f:
    _f.write(f"\n{'='*60}\n")
    _f.write(f"Run date/time  : {run_end.strftime('%Y-%m-%d %H:%M:%S')}\n")
    _f.write(f"Results file   : {csv_path.name}\n")
    _f.write(f"Conditions     : {CONDITIONS}\n")
    _f.write(f"File mode      : {FILE_MODE}\n")
    if FILE_MODE == "ome_tiff":
        _f.write(f"Ref channel    : {REF_CHANNEL_IDX} (ch{REF_CHANNEL_IDX+1}, baseline)\n")
        _f.write(f"Sig channel    : {SIG_CHANNEL_IDX} (ch{SIG_CHANNEL_IDX+1}, expression)\n")
    _f.write(f"Model          : {CELLPOSE_MODEL}\n")
    _f.write(f"Diameter       : {CELL_DIAMETER}\n")
    _f.write(f"Flow thresh    : {FLOW_THRESHOLD}\n")
    _f.write(f"CellProb thresh: {CELLPROB_THRESHOLD}\n")
    _f.write(f"Area filter    : {MIN_CELL_AREA_PX}–{MAX_CELL_AREA_PX} px²\n")
    _f.write(f"Total cells    : {len(df_all)}\n")
print(f"Run log appended to {log_path}")